# Reverse Jawa ke Indonesia, lengan semua 20 epoch (VALID saja, test tidak dinilai)

M2M100-418M, 15 epoch, batch 16, lr 5e-5, seed 42; tanpa tag, oversampling, juri, dan rerank.

| Lengan | Data |
|---|---|
| `base` | Unggah-Ungguh train 3 tingkat |
| `semua` | + NgokoKrama, NusaX, NusaTranslation |

Checkpoint = chrF++ greedy valid tertinggi. Angka akhir = beam 5. Pembanding: model 5 epoch pool_makna_v1 50,41 / BLEU 30,57.
Aturan baca (dikunci sebelum push): `semua` menang jika chrF++ beam 5 lebih tinggi dari base DAN tidak ada tingkat input yang
turun > 1,0; selain itu base. Langkah ikut jumlah data, jadi efek data dan langkah belum terpisah. 1 seed.
Weights `semua` dilatih NgokoKrama (tanpa lisensi): jangan dibagikan. Data eksternal tidak disimpan di output.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece", "openpyxl", "scikit-learn"], check=False)

import glob, json, time, random, re, os, hashlib, gc, shutil
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu, transformers
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

REPO, SL, TL = "facebook/m2m100_418M", "jv", "id"
TRAIN_SEED = 42
EPOCHS, OS, BS, LR, WARMUP, JUDGE_SEED, MAXLEN, CTRL_W = 20, 3, 16, 5e-5, 0.1, 42, 64, 30.0
NBEST, LAM = 8, 2.0
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
TAG = {l: f"<{l}>" for l in LEVELS}
INIT_WORDS = {"ngoko": "kowe aku mangan lunga turu", "krama": "sampeyan kula nedha kesah tilem",
              "krama_alus": "panjenenganipun dalem dhahar tindak sare"}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


find = lambda n: sorted(p for p in glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True) if "/output" not in p)[0]
tr_raw = pd.read_csv(find("train"))
def _load3(n):
    d = pd.read_csv(find(n))
    d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)
tr, va, te = (_load3(n) for n in ["train", "valid", "test"])
assert (len(tr), len(va), len(te)) == (1395, 175, 176), (len(tr), len(va), len(te))
log("data", len(tr), len(va), len(te), torch.cuda.get_device_name(0))
tag_src = lambda ind, level: f"{TAG[level]} {ind}"
def bleu(h, r): return sacrebleu.corpus_bleu(h, [r]).score
def chrf(h, r): return sacrebleu.corpus_chrf(h, [r], word_order=2).score
MANIFEST = {"protokol": "reverse jv-id ep20 semua", "train_seed": TRAIN_SEED,
            "hparams": {"epochs": EPOCHS, "bs": BS, "lr": LR, "warmup": WARMUP, "maxlen": MAXLEN, "beams": 5},
            "versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                         "transformers": transformers.__version__, "sacrebleu": sacrebleu.__version__},
            "gpu": torch.cuda.get_device_name(0),
            "inputs": {n: sha(find(n)) for n in ["train", "valid", "test"]},
            "runs": {}}

In [ ]:
SKRIP = r'''"""Gabungkan Unggah-Ungguh (train saja) + NgokoKrama + NusaTranslation + NusaX untuk 3 tingkat.

Jalankan dari root repo:  python scripts/gabung_data_eksternal.py
Input : data/external/{ngokokrama/NgokoKrama.xlsx, nusatranslation/jav-*.csv, nusax/train.csv} (di-gitignore)
Output: data/merged/train_eksternal_3lv.csv, data/merged/train_eksternal_3lv_laporan.json (di-gitignore)

- Valid/test Unggah-Ungguh tidak disentuh. Baris eksternal yang mirip valid/test dibuang.
- Label tingkat baris eksternal ditentukan classifier TF-IDF 4 tingkat yang dilatih di train Unggah-Ungguh.
  Sengaja bukan juri DistilBERT / reranker RoBERTa, supaya evaluasi tidak sirkular.
- "Krama" NgokoKrama hanya boleh menjadi krama atau krama_alus; "ngoko" hanya ngoko. Selain itu dibuang.
- NgokoKrama tidak berlisensi: pakai lokal saja, jangan disebarkan. Nonaktifkan dengan --tanpa-ngokokrama.
"""
import json
import re
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

LEVELS = ["ngoko", "krama", "krama_alus"]
MIN_KATA, MAX_KATA = 3, 25
P_MIN = {"ngoko": 0.60, "krama": 0.60, "krama_alus": 0.85}  # KA < 0,85 sering keliru (audit sampel)
MIRIP = 0.80          # ambang cosine dengan valid/test
CAP_NUSATRANS = 300  # supaya ngoko tidak menenggelamkan tingkat lain
SEED = 42


def norm(s):
    s = str(s).replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    return re.sub(r"\s+", " ", s).strip()


def kunci(s):
    return re.sub(r"\W+", " ", str(s).lower()).strip()


def split(n):
    return pd.read_csv(f"data/unggah_ungguh/split/{n}.csv")


tr, va, te = split("train"), split("valid"), split("test")
clf = make_pipeline(
    make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2),
               TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    LogisticRegression(C=20, max_iter=3000, class_weight="balanced"))
clf.fit(tr["javanese"], tr["level"])
lap = {"akurasi_pelabel_di_valid": {}}
pv, kv = clf.predict(va["javanese"]), clf.predict_proba(va["javanese"]).max(1)
for l in clf.classes_:
    yakin = (pv == l) & (kv >= P_MIN.get(l, 0.60))
    lap["akurasi_pelabel_di_valid"][l] = {
        "recall": round(float((pv[va.level == l] == l).mean()), 3),
        "presisi_saat_yakin": round(float((va.level[yakin] == l).mean()), 3), "n_yakin": int(yakin.sum())}

rows = []
if "--tanpa-ngokokrama" not in sys.argv:
    nk = pd.read_excel("data/external/ngokokrama/NgokoKrama.xlsx", sheet_name="All")
    for c, peran in [("javanese ngoko", "ngoko"), ("javanese krama", "krama")]:
        rows.append(pd.DataFrame({"indonesian": nk["indonesian"], "javanese": nk[c], "source": "ngokokrama", "peran": peran}))
for s in ["train", "valid", "test"]:
    d = pd.read_csv(f"data/external/nusatranslation/jav-{s}.csv")
    rows.append(pd.DataFrame({"indonesian": d["ind_text"], "javanese": d["tgt_text"], "source": "nusatranslation", "peran": "umum"}))
d = pd.read_csv("data/external/nusax/train.csv")
rows.append(pd.DataFrame({"indonesian": d["indonesian"], "javanese": d["javanese"], "source": "nusax", "peran": "umum"}))
x = pd.concat(rows, ignore_index=True).dropna()
buang = {}


def saring(mask, alasan):
    global x
    buang[alasan] = {k: int(v) for k, v in x[~mask].groupby("source").size().items()}
    x = x[mask].reset_index(drop=True)


for c in ["indonesian", "javanese"]:
    x[c] = x[c].map(norm)
lap["awal"] = x.groupby(["source", "peran"]).size().rename(lambda t: t, axis=0).to_dict()
lap["awal"] = {"/".join(k): int(v) for k, v in lap["awal"].items()}
ni, nj = x.indonesian.str.split().str.len(), x.javanese.str.split().str.len()
saring(ni.between(MIN_KATA, MAX_KATA) & nj.between(MIN_KATA, MAX_KATA) & (ni / nj).between(0.6, 1.6), "panjang")
noise = r"https?:|www\.|\[url\]|\[username\]|@\w|#\w|[^\x00-\x7FéèêÉÈ–—…]"
saring(~(x.indonesian.str.contains(noise, regex=True, case=False) | x.javanese.str.contains(noise, regex=True, case=False)), "url_mention_emoji")
saring(x.indonesian.map(kunci) != x.javanese.map(kunci), "sama_dengan_sumber")

ev = pd.concat([va, te])
tolak = np.zeros(len(x), bool)
for c in ["indonesian", "javanese"]:
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4)).fit(pd.concat([ev[c], x[c]]))
    sim = np.asarray((v.transform(x[c]) @ v.transform(ev[c]).T).max(1).todense()).ravel()
    lap[f"kemiripan_maks_{c}"] = round(float(sim.max()), 3)
    tolak |= sim >= MIRIP
saring(pd.Series(~tolak), "mirip_valid_test")

P = clf.predict_proba(x["javanese"])
x["level"], x["p_level"] = clf.classes_[P.argmax(1)], P.max(1).round(3)
boleh = {"ngoko": ["ngoko"], "krama": ["krama", "krama_alus"], "umum": ["ngoko"]}
saring(pd.Series([l in boleh[p] for l, p in zip(x.level, x.peran)]), "tingkat_tidak_cocok")
saring(x.p_level >= x.level.map(P_MIN), "label_tidak_yakin")

uu = tr[tr.level.isin(LEVELS)].assign(source="unggah_ungguh", p_level=1.0)
sudah = set(zip(uu.indonesian.map(kunci), uu.javanese.map(kunci)))
x["k"] = list(zip(x.indonesian.map(kunci), x.javanese.map(kunci)))
saring(~x.k.duplicated() & ~x.k.isin(sudah), "duplikat")
nt = x[x.source == "nusatranslation"]
if len(nt) > CAP_NUSATRANS:
    saring(~x.index.isin(nt.drop(nt.sample(CAP_NUSATRANS, random_state=SEED).index).index), "cap_nusatranslation")

x["group"] = 100000 + pd.factorize(x.indonesian.map(kunci))[0]
kol = ["group", "level", "indonesian", "javanese", "source", "p_level"]
out = pd.concat([uu[kol], x[kol]], ignore_index=True)
out.to_csv("data/merged/train_eksternal_3lv.csv", index=False, encoding="utf-8")
tab = out.groupby(["source", "level"]).size().unstack(fill_value=0)
for n, e in [("valid", va), ("test", te)]:
    lap[f"bocor_persis_{n}"] = {c: len(set(x[c].map(kunci)) & set(e[c].map(kunci))) for c in ["indonesian", "javanese"]}
lap.update(dibuang=buang, hasil={s: r.to_dict() for s, r in tab.iterrows()}, total={k: int(v) for k, v in tab.sum().items()})
json.dump(lap, open("data/merged/train_eksternal_3lv_laporan.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False, default=int)
print(json.dumps(lap, indent=1, ensure_ascii=False, default=int)); print(tab)
'''

import urllib.request
W = "/tmp/eksternal"
for d in ["data/unggah_ungguh/split", "data/external/ngokokrama", "data/external/nusatranslation", "data/external/nusax", "data/merged"]:
    os.makedirs(f"{W}/{d}", exist_ok=True)
for n in ["train", "valid", "test"]:
    shutil.copy(find(n), f"{W}/data/unggah_ungguh/split/{n}.csv")


def unduh(u, p):
    for k in range(4):
        try:
            return urllib.request.urlretrieve(u, f"{W}/data/external/{p}")
        except Exception as e:
            log("unduh gagal", p, repr(e)); time.sleep(10)
    raise RuntimeError(p)


RAW = "https://raw.githubusercontent.com"
unduh(f"{RAW}/ir-nlp-csui/NgokoKrama/HEAD/dataset/NgokoKrama.xlsx", "ngokokrama/NgokoKrama.xlsx")
for s in ["train", "valid", "test"]:
    unduh(f"{RAW}/IndoNLP/nusa-writes/main/data/nusa_kalimat-mt-jav-{s}.csv", f"nusatranslation/jav-{s}.csv")
unduh(f"{RAW}/IndoNLP/nusax/main/datasets/mt/train.csv", "nusax/train.csv")
open(f"{W}/gabung.py", "w", encoding="utf-8").write(SKRIP)
p = subprocess.run([sys.executable, "gabung.py"], cwd=W, capture_output=True, text=True, env={**os.environ, "PYTHONUTF8": "1"})
print(p.stdout[-6000:], p.stderr[-3000:]); assert p.returncode == 0
EXT = pd.read_csv(f"{W}/data/merged/train_eksternal_3lv.csv")
MANIFEST["data_eksternal"] = json.load(open(f"{W}/data/merged/train_eksternal_3lv_laporan.json", encoding="utf-8"))
for k in ["bocor_persis_valid", "bocor_persis_test"]:
    assert all(v == 0 for v in MANIFEST["data_eksternal"][k].values()), k
EXT = EXT[EXT["source"] != "unggah_ungguh"].reset_index(drop=True)
shutil.rmtree(W)
log("eksternal", EXT.groupby(["source", "level"]).size().to_dict(), "| baris lokal 2192, didapat", len(EXT))


In [ ]:
def translate(tok, model, bos, srcs, beams=4, bs=48):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=MAXLEN)
        for j, t in zip(idx, tok.batch_decode(g, skip_special_tokens=True)):
            out[j] = t
    return out


def metrik(h):
    res = {}
    for lv in ["all"] + LEVELS:
        ix = [i for i, l in enumerate(va["level"]) if lv in ("all", l)]
        a, r = [h[i] for i in ix], [va["indonesian"].iloc[i] for i in ix]
        res[lv] = {"n": len(a), "chrF++": round(chrf(a, r), 2), "BLEU": round(bleu(a, r), 2)}
    return res


def train_run(name, extra):
    seed_all(TRAIN_SEED)
    tok = AutoTokenizer.from_pretrained(REPO, src_lang=SL, tgt_lang=TL)
    model = AutoModelForSeq2SeqLM.from_pretrained(REPO, dtype=torch.float32).to(dev)
    bos = tok.get_lang_id(TL); va_src = va["javanese"].tolist()
    d = pd.concat([tr, extra])[["javanese", "indonesian"]].drop_duplicates()
    pairs = d.values.tolist()
    nb = (len(pairs) + BS - 1) // BS; steps = EPOCHS * nb
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    best, best_state, hist = -1, None, []
    for ep in range(1, EPOCHS + 1):
        model.train(); random.shuffle(pairs); tot = 0.0
        for i in range(0, len(pairs), BS):
            b = pairs[i:i + BS]
            enc = tok([s for s, _ in b], text_target=[t for _, t in b], return_tensors="pt",
                      padding=True, truncation=True, max_length=MAXLEN).to(dev)
            labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc, labels=labels).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(b)
        c = chrf(translate(tok, model, bos, va_src, 1), va["indonesian"].tolist())
        hist.append({"epoch": ep, "train_loss": round(tot / len(pairs), 4), "valid_chrF++": round(c, 2)}); log(name, hist[-1])
        if c > best:
            best = c; best_state = {k: x.detach().to("cpu", copy=True) for k, x in model.state_dict().items()}
    del opt, scaler, sched; gc.collect(); torch.cuda.empty_cache()
    model.load_state_dict(best_state); del best_state; gc.collect()
    hyp = translate(tok, model, bos, va_src, 5, 16)
    pv = va[["index", "group", "level", "javanese", "indonesian"]].copy(); pv["hyp"] = hyp
    pv.to_csv(f"{OUT}/valid_{name}.csv", index=False)
    res = {"n_pairs": len(pairs), "steps": steps, "history": hist, "best_epoch": max(hist, key=lambda h: h["valid_chrF++"])["epoch"],
           "valid_beam5": metrik(hyp), "sama_persis": sum(x.strip() == y.strip() for x, y in zip(hyp, va["indonesian"]))}
    model.half().save_pretrained(f"{OUT}/model_{name}"); tok.save_pretrained(f"{OUT}/model_{name}")
    del model; gc.collect(); torch.cuda.empty_cache()
    log(name, res["valid_beam5"]); return res


In [ ]:
simpan = lambda: json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1, default=str)
for name, ex in [("semua", EXT)]:
    try:
        MANIFEST["runs"][name] = train_run(name, ex)
    except Exception as e:
        import traceback; traceback.print_exc()
        MANIFEST["runs"][name] = {"error": repr(e)}; gc.collect(); torch.cuda.empty_cache()
    simpan()
R = {k: v["valid_beam5"] for k, v in MANIFEST["runs"].items() if "valid_beam5" in v}
if len(R) == 2:
    b, s = R["base"], R["semua"]
    MANIFEST["pemenang"] = "semua" if s["all"]["chrF++"] > b["all"]["chrF++"] and all(
        s[l]["chrF++"] >= b[l]["chrF++"] - 1.0 for l in LEVELS) else "base"
MANIFEST["elapsed_s"] = round(time.time() - T0); simpan()
print(json.dumps(R, indent=1)); log("PEMENANG", MANIFEST.get("pemenang")); log("SELESAI")
